## 전역

In [3]:
# ============================================================
# Top10 1위 조합 기반 SHAP + Permutation Importance 분석
# (Top10_우수모델.csv의 FeatureFile / Method / SMOTE_Ratio 사용)
# ============================================================

import os
import platform
import warnings
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

# 불균형 처리 방식(Method)에 따라 필요한 라이브러리 (없으면 해당 방식만 사용 불가)
try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")

# ── ★ 한글 폰트 설정 ──────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    try:
        import koreanize_matplotlib
    except ImportError:
        pass

plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. Top10_우수모델.csv 1번째 행 정보 로드
# ============================================================
TOP10_PATH = "15번. 우수모델 데이터/Top10 우수모델.csv"

top10 = pd.read_csv(TOP10_PATH, index_col=0)
row = top10.iloc[0]   # 1위 조합

FEATURE_SET  = row["FeatureSet"]
FEATURE_FILE = row["FeatureFile"]
METHOD       = row["Method"]
SMOTE_RATIO  = row["SMOTE_Ratio"]
MODEL_NAME   = row["Model"]

print("=" * 70)
print("Top10 1위 조합")
print(f"  FeatureSet  : {FEATURE_SET}")
print(f"  FeatureFile : {FEATURE_FILE}")
print(f"  Method      : {METHOD}")
print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
print(f"  Model       : {MODEL_NAME}")
print("=" * 70)


# ============================================================
# 2. 경로 / 설정값
# ============================================================
TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = os.path.join(r'13번.피처셀렉션\M19_도매_소매업', FEATURE_FILE)

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42

SHAP_SAVE_DIR = r"16번. SHAP\전역"
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)


# ============================================================
# 3. Method / SMOTE_Ratio 기반 불균형 처리 함수
#   (이전 단계 run_top1_recall09.py 와 동일한 로직)
# ============================================================
def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    minority   = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()

    target_n = int(n_majority * ratio) if ratio else n_majority
    n_synth  = max(0, target_n - n_minority)

    if n_synth == 0 or n_minority < 5:
        return X, y

    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    """
    Method/SMOTE_Ratio에 따라 (X_res, y_res, pos_weight)를 반환.
    - 리샘플링이 적용되면 클래스가 균형화되므로 pos_weight=1.0
    - 리샘플링이 없으면(ClassWeight) pos_weight = (다수/소수)
    """
    method_l = str(method).strip().lower()
    ratio = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("BorderlineSMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("SMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' -> ClassWeight(pos_weight)로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 4. 데이터 로드 + 피처 로드
# ============================================================
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

print(f"\n피처 수: {len(use_features)}개")
print("=" * 70)


# ============================================================
# 5. 모델 정의 (XGBoost, 주어진 하이퍼파라미터 + Method별 pos_weight)
# ============================================================
def make_final_model(pos_weight):
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 6. 전체 train 재학습 (Method/SMOTE_Ratio 기반 리샘플링 적용)
# ============================================================
print(f"\n{'='*70}")
print(f"[SHAP/Permutation] FeatureSet: {FEATURE_SET}  |  피처 수: {len(use_features)}개")
print(f"{'='*70}")

imputer_shap = SimpleImputer(strategy="median")
X_train_shap = pd.DataFrame(
    imputer_shap.fit_transform(train_full[use_features]),
    columns=use_features
)
X_test_shap  = pd.DataFrame(
    imputer_shap.transform(test[use_features]),
    columns=use_features
)

X_train_res, y_train_res, pos_weight = apply_resampling(
    X_train_shap, y_train_full, METHOD, SMOTE_RATIO
)
print(f"  리샘플링 적용({METHOD}, SMOTE_Ratio={SMOTE_RATIO}): "
      f"{len(X_train_shap)}행 -> {len(X_train_res)}행, pos_weight={pos_weight:.4f}")

model_shap = make_final_model(pos_weight)
model_shap.fit(X_train_res, y_train_res)

# ── 서브폴더: 16번. SHAP/전역/{FeatureSet} ───────────────────
save_sub = os.path.join(SHAP_SAVE_DIR, FEATURE_SET)
os.makedirs(save_sub, exist_ok=True)


# ============================================================
# 7. [A-1] SHAP 계산
# ============================================================
print(f"\n  SHAP 계산 중...")
explainer   = shap.TreeExplainer(model_shap)
shap_values = explainer.shap_values(X_test_shap)

shap_df = pd.DataFrame(shap_values, columns=use_features)
shap_df.to_csv(
    os.path.join(save_sub, f"shap_values_{FEATURE_SET}.csv"),
    index=False, encoding="utf-8-sig"
)

# ── [A-2] Bar Plot ────────────────────────────────────────
plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
shap.summary_plot(shap_values, X_test_shap, plot_type="bar",
                  show=False, max_display=20)
plt.title(f"SHAP Global Feature Importance (Bar)\n{FEATURE_SET} ({METHOD})", fontsize=12)
plt.tight_layout()
plt.savefig(
    os.path.join(save_sub, f"shap_bar_{FEATURE_SET}.png"),
    dpi=150, bbox_inches="tight"
)
plt.close()

# ── [A-3] Beeswarm Plot ───────────────────────────────────
plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
shap.summary_plot(shap_values, X_test_shap, plot_type="dot",
                  show=False, max_display=20)
plt.title(f"SHAP Beeswarm Plot\n{FEATURE_SET} ({METHOD})", fontsize=12)
plt.tight_layout()
plt.savefig(
    os.path.join(save_sub, f"shap_beeswarm_{FEATURE_SET}.png"),
    dpi=150, bbox_inches="tight"
)
plt.close()

# ── [A-4] SHAP 중요도 CSV ─────────────────────────────────
shap_importance = pd.DataFrame({
    "Feature"       : use_features,
    "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
shap_importance["Rank"] = shap_importance.index + 1
shap_importance.to_csv(
    os.path.join(save_sub, f"shap_importance_{FEATURE_SET}.csv"),
    index=False, encoding="utf-8-sig"
)
print(f"  [SHAP Top 10]\n{shap_importance.head(10).to_string(index=False)}")


# ============================================================
# 8. [B-1] Permutation Importance
# ============================================================
print(f"\n  Permutation Importance 계산 중...")
perm_result = permutation_importance(
    model_shap, X_test_shap, y_test,
    n_repeats=30, random_state=RANDOM_STATE,
    scoring="average_precision", n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature"   : use_features,
    "Perm_Mean" : perm_result.importances_mean,
    "Perm_Std"  : perm_result.importances_std,
}).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
perm_df["Rank"] = perm_df.index + 1
perm_df.to_csv(
    os.path.join(save_sub, f"permutation_importance_{FEATURE_SET}.csv"),
    index=False, encoding="utf-8-sig"
)
print(f"  [Permutation Top 10]\n{perm_df.head(10).to_string(index=False)}")

# ── [B-2] Permutation Bar Plot ────────────────────────────
top_n    = min(20, len(use_features))
perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
ax.barh(
    perm_top["Feature"], perm_top["Perm_Mean"],
    xerr=perm_top["Perm_Std"],
    color="#2F6EBA", alpha=0.8,
    error_kw=dict(ecolor="#555555", capsize=3)
)
ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
ax.set_title(f"Permutation Importance (Top {top_n})\n{FEATURE_SET} ({METHOD})", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(
    os.path.join(save_sub, f"permutation_bar_{FEATURE_SET}.png"),
    dpi=150, bbox_inches="tight"
)
plt.close()


# ============================================================
# 9. [C] SHAP vs Permutation 순위 비교 CSV
# ============================================================
compare_df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
    columns={"Rank": "SHAP_Rank"}
).merge(
    perm_df[["Feature", "Perm_Mean", "Rank"]].rename(
        columns={"Rank": "Perm_Rank"}
    ),
    on="Feature", how="inner"
)
compare_df["Rank_Diff"] = (compare_df["SHAP_Rank"] - compare_df["Perm_Rank"]).abs()
compare_df = compare_df.sort_values("SHAP_Rank").reset_index(drop=True)
compare_df.to_csv(
    os.path.join(save_sub, f"shap_vs_permutation_{FEATURE_SET}.csv"),
    index=False, encoding="utf-8-sig"
)

print(f"\n  → {save_sub} 저장 완료")
print("\n" + "=" * 70)
print("SHAP / Permutation Importance 분석 완료 (Top10 1위 조합 기준)")
print(f"  FeatureSet: {FEATURE_SET} | Method: {METHOD} | SMOTE_Ratio: {SMOTE_RATIO}")
print(f"  저장 위치: {save_sub}")
print("=" * 70)

Top10 1위 조합
  FeatureSet  : top65_dedup52
  FeatureFile : lasso_features_top65--52.csv
  Method      : ClassWeight
  SMOTE_Ratio : -
  Model       : XGBoost

피처 수: 52개

[SHAP/Permutation] FeatureSet: top65_dedup52  |  피처 수: 52개
  리샘플링 적용(ClassWeight, SMOTE_Ratio=-): 28111행 -> 28111행, pos_weight=25.6455

  SHAP 계산 중...
  [SHAP Top 10]
                Feature  mean_abs_SHAP  Rank
          총자본영업이익률_diff       1.459370     1
            영업이익률_ratio       1.342260     2
                금융비용부담률       0.748039     3
                  ROA변화       0.666008     4
금융비용대매출액_ratio_industry       0.434247     5
  매출액순이익률_diff_industry       0.352926     6
                  자본잠식률       0.306432     7
              ROIC_diff       0.260909     8
              ROA_ratio       0.215919     9
             순이익률_ratio       0.191534    10

  Permutation Importance 계산 중...
  [Permutation Top 10]
                Feature  Perm_Mean  Perm_Std  Rank
                  ROA변화   0.186345  0.008759     1
          

In [4]:
# ============================================================
# SHAP vs Permutation 통합 분석 CSV 생성 (피처 범주 추가)
# ============================================================

import pandas as pd
import numpy as np
import os

SHAP_SAVE_DIR = r"16번. SHAP\전역"

# ============================================================
# ★ 피처 범주 사전 정의
# ============================================================

FEATURE_CATEGORY = {
    # 안정성 (Solvency)
    "자본잠식률"                        : "안정성 (Solvency)",
    "비유동장기적합률_ratio"             : "안정성 (Solvency)",
    "차입금의존도_diff_industry"         : "안정성 (Solvency)",
    "부채비율"                          : "안정성 (Solvency)",
    "자기자본비율_diff_industry"         : "안정성 (Solvency)",
    "부채비율변화"                       : "안정성 (Solvency)",
    "장기부채의존도"                     : "안정성 (Solvency)",
    "유동비율변화_diff"                  : "안정성 (Solvency)",
    "유동비율_ratio"                     : "안정성 (Solvency)",
    "장기부채비율"                       : "안정성 (Solvency)",
    "유보율_diff"                        : "안정성 (Solvency)",
    "순운전자본비율_ratio"               : "안정성 (Solvency)",
    "순운전자본대총자본_ratio_industry"  : "안정성 (Solvency)",

    # 수익성 (Profitability)
    "총자본영업이익률_diff"              : "수익성 (Profitability)",
    "금융비용부담률"                     : "수익성 (Profitability)",
    "ROA변화"                           : "수익성 (Profitability)",
    "매출액순이익률_diff_industry"       : "수익성 (Profitability)",
    "ROIC_diff"                         : "수익성 (Profitability)",
    "ROA_ratio"                         : "수익성 (Profitability)",
    "순이익률_ratio"                     : "수익성 (Profitability)",
    "현금ROA"                           : "수익성 (Profitability)",
    "매출총이익률_diff"                  : "수익성 (Profitability)",
    "매출원가율"                         : "수익성 (Profitability)",
    "ROE_diff"                          : "수익성 (Profitability)",
    "현금ROE_ratio"                     : "수익성 (Profitability)",

    # 성장성 (Growth)
    "매출액증가율"                       : "성장성 (Growth)",
    "순이익증가율_diff"                  : "성장성 (Growth)",
    "유형자산증가율_ratio_industry"      : "성장성 (Growth)",
    "총자산증가율_diff_industry"         : "성장성 (Growth)",
    "자기자본증가율"                     : "성장성 (Growth)",

    # 활동성 (Activity)
    "비유동자산회전율_ratio"             : "활동성 (Activity)",
    "유형자산회전율_diff"                : "활동성 (Activity)",
    "매출채권회전율_ratio_industry"      : "활동성 (Activity)",
    "순운전자본회전율_diff"              : "활동성 (Activity)",
    "유동자산회전율"                     : "활동성 (Activity)",
    "총자산회전율_ratio"                 : "활동성 (Activity)",
    "재고자산보유기간_ratio"             : "활동성 (Activity)",
    "매입채무지급기간_diff"              : "활동성 (Activity)",

    # 현금흐름 (Cash Flow)
    "영업CF_유동부채_diff"               : "현금흐름 (Cash Flow)",
    "영업CF_총부채_diff"                 : "현금흐름 (Cash Flow)",
    "FCF_총자산_ratio"                   : "현금흐름 (Cash Flow)",
    "영업현금흐름비율"                   : "현금흐름 (Cash Flow)",
    "감가상각비율"                       : "현금흐름 (Cash Flow)",

    # 기타
    "업력"                              : "기타 (Other)",
    "유형자산비율"                       : "기타 (Other)",
}

CATEGORY_ORDER = {
    "안정성 (Solvency)"        : 1,
    "수익성 (Profitability)"   : 2,
    "성장성 (Growth)"          : 3,
    "활동성 (Activity)"        : 4,
    "현금흐름 (Cash Flow)"     : 5,
    "기타 (Other)"             : 6,
    "미분류"                   : 7,
}


# ============================================================
# 피처셋별 루프
# ============================================================

for feature_name in feature_map.keys():

    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)

    # ── 파일 로드 ─────────────────────────────────────────
    shap_imp = pd.read_csv(
        os.path.join(save_sub, f"shap_importance_{feature_name}.csv")
    )
    perm_imp = pd.read_csv(
        os.path.join(save_sub, f"permutation_importance_{feature_name}.csv")
    )

    n_features = len(shap_imp)

    # ── 병합 ──────────────────────────────────────────────
    df = shap_imp[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_imp[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )

    # ── 정규화 순위 ───────────────────────────────────────
    df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / (n_features - 1)
    df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / (n_features - 1)

    # ── 순위 차이 ─────────────────────────────────────────
    df["Rank_Diff"] = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()

    # ── 통합 중요도 점수 ──────────────────────────────────
    df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
    df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

    # ── 피처 유형 분류 ────────────────────────────────────
    top_n    = max(3, int(n_features * 0.3))
    shap_top = set(df.nsmallest(top_n, "SHAP_Rank")["Feature"])
    perm_top = set(df.nsmallest(top_n, "Perm_Rank")["Feature"])

    def classify(row):
        in_shap = row["Feature"] in shap_top
        in_perm = row["Feature"] in perm_top
        if in_shap and in_perm:
            return "★ 핵심피처 (SHAP+Perm 모두 높음)"
        elif in_shap and not in_perm:
            return "△ 대체가능 (SHAP 높음, Perm 낮음)"
        elif not in_shap and in_perm:
            return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
        else:
            return "- 일반피처"

    df["Feature_Type"] = df.apply(classify, axis=1)

    # ── 불일치 레벨 ───────────────────────────────────────
    def rank_diff_level(diff):
        if diff <= 3:
            return "일치"
        elif diff <= 8:
            return "소폭 불일치"
        else:
            return "대폭 불일치"

    df["Consistency"] = df["Rank_Diff"].apply(rank_diff_level)

    # ── 범주 컬럼 추가 ────────────────────────────────────
    df["Category"]       = df["Feature"].map(FEATURE_CATEGORY).fillna("미분류")
    df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

    # ── 컬럼 순서 정리 & 정렬 ─────────────────────────────
    df = df[[
        "Combined_Rank",
        "Feature",
        "Category",
        "Feature_Type",
        "SHAP_Rank", "mean_abs_SHAP",
        "Perm_Rank", "Perm_Mean", "Perm_Std",
        "Rank_Diff", "Consistency",
        "Combined_Score",
        "Category_Order",
    ]].sort_values("Combined_Rank").reset_index(drop=True)

    # ── 저장 ──────────────────────────────────────────────
    out_path = os.path.join(save_sub, f"feature_analysis_{feature_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    # ── 요약 출력 ─────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"[{feature_name}]  피처 분석 완료  →  {out_path}")
    print(f"{'='*70}")

    # 피처 유형 분포
    print(f"\n  [피처 유형 분포]")
    for t, c in df["Feature_Type"].value_counts().items():
        print(f"    {t} : {c}개")

    # ── ★ 범주별 분포 (수정된 부분) ──────────────────────
    print(f"\n  [범주별 분포]")
    cat_order_df = df[["Category", "Category_Order"]].drop_duplicates()

    cat_summary = (
        df.groupby("Category")
        .agg(
            피처수             = ("Feature",      "count"),
            평균_Combined_Rank = ("Combined_Rank", "mean"),
            핵심피처수         = ("Feature_Type",
                                  lambda x: (x == "★ 핵심피처 (SHAP+Perm 모두 높음)").sum())
        )
        .reset_index()
        .merge(cat_order_df, on="Category", how="left")   # ★ merge 먼저
        .sort_values("Category_Order")                    # ★ 그 다음 정렬
        .reset_index(drop=True)
    )
    print(cat_summary[[
        "Category", "피처수", "평균_Combined_Rank", "핵심피처수"
    ]].to_string(index=False))

    # 핵심피처 목록
    print(f"\n  [★ 핵심피처 목록 (SHAP+Perm 모두 상위 {top_n}위 이내)]")
    core = df[df["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"]
    if len(core) > 0:
        print(core[[
            "Combined_Rank", "Feature", "Category",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

    # 미분류 피처 경고
    unclassified = df[df["Category"] == "미분류"]
    if len(unclassified) > 0:
        print(f"\n  [⚠️ 미분류 피처 ({len(unclassified)}개) — FEATURE_CATEGORY 사전에 추가 필요]")
        print(unclassified[["Feature", "SHAP_Rank", "Perm_Rank"]].to_string(index=False))

    # 대폭 불일치 피처
    print(f"\n  [대폭 불일치 피처 (Rank_Diff > 8)]")
    mismatch = df[df["Consistency"] == "대폭 불일치"].sort_values(
        "Rank_Diff", ascending=False
    )
    if len(mismatch) > 0:
        print(mismatch[[
            "Feature", "Category", "Feature_Type",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

print("\n" + "=" * 70)
print("전체 피처 분석 CSV 저장 완료")
print("=" * 70)


[top55_dedup45]  피처 분석 완료  →  16번. SHAP\전역\top55_dedup45\feature_analysis_top55_dedup45.csv

  [피처 유형 분포]
    - 일반피처 : 28개
    ★ 핵심피처 (SHAP+Perm 모두 높음) : 9개
    ▲ 상호작용 (Perm 높음, SHAP 낮음) : 4개
    △ 대체가능 (SHAP 높음, Perm 낮음) : 4개

  [범주별 분포]
           Category  피처수  평균_Combined_Rank  핵심피처수
     안정성 (Solvency)   13         23.307692      1
수익성 (Profitability)   12         19.666667      6
       성장성 (Growth)    5         23.200000      1
     활동성 (Activity)    8         25.625000      0
   현금흐름 (Cash Flow)    5         25.400000      1
         기타 (Other)    2         21.500000      0

  [★ 핵심피처 목록 (SHAP+Perm 모두 상위 13위 이내)]
 Combined_Rank               Feature            Category  SHAP_Rank  Perm_Rank  Rank_Diff
             1         총자본영업이익률_diff 수익성 (Profitability)          1          1          0
             2               금융비용부담률 수익성 (Profitability)          2          3          1
             2                 ROA변화 수익성 (Profitability)          3          2          1
          

## 지역

In [6]:
# ============================================================
# Top10 1위 조합 기반 로컬 SHAP(Waterfall) 분석
# (Top10_우수모델.csv의 FeatureFile / Method / SMOTE_Ratio 사용)
# ============================================================

import os
import platform
import warnings
import numpy as np
import pandas as pd
import shap
import matplotlib
import matplotlib as mpl
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score
from xgboost import XGBClassifier

# 불균형 처리 방식(Method)에 따라 필요한 라이브러리 (없으면 해당 방식만 사용 불가)
try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")

# ── 한글 폰트 ─────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
    mpl.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
    mpl.rcParams["font.family"] = "AppleGothic"
else:
    try:
        import koreanize_matplotlib
    except ImportError:
        pass

plt.rcParams["axes.unicode_minus"] = False
mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. Top10_우수모델.csv 1번째 행 정보 로드
# ============================================================
TOP10_PATH = "15번. 우수모델 데이터/Top10 우수모델.csv"

top10 = pd.read_csv(TOP10_PATH, index_col=0)
row = top10.iloc[0]   # 1위 조합

FEATURE_SET  = row["FeatureSet"]
FEATURE_FILE = row["FeatureFile"]
METHOD       = row["Method"]
SMOTE_RATIO  = row["SMOTE_Ratio"]
MODEL_NAME   = row["Model"]

print("=" * 70)
print("Top10 1위 조합")
print(f"  FeatureSet  : {FEATURE_SET}")
print(f"  FeatureFile : {FEATURE_FILE}")
print(f"  Method      : {METHOD}")
print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
print(f"  Model       : {MODEL_NAME}")
print("=" * 70)


# ============================================================
# 2. 경로 / 설정값
# ============================================================
TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = os.path.join(r'13번.피처셀렉션\M19_도매_소매업', FEATURE_FILE)

TARGET_COL   = "부실라벨_ICR3년"
YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
RANDOM_STATE = 42
RECALL_MIN   = 0.9   # Test 기준 threshold 산출 조건 (run_top1_recall09.py와 동일)

SHAP_SAVE_DIR = r"16번. SHAP\지역"
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)


# ============================================================
# 3. Method / SMOTE_Ratio 기반 불균형 처리 함수
#   (run_top1_recall09.py / run_top1_shap.py 와 동일한 로직)
# ============================================================
def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    minority   = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()

    target_n = int(n_majority * ratio) if ratio else n_majority
    n_synth  = max(0, target_n - n_minority)

    if n_synth == 0 or n_minority < 5:
        return X, y

    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    """
    Method/SMOTE_Ratio에 따라 (X_res, y_res, pos_weight)를 반환.
    - 리샘플링이 적용되면 클래스가 균형화되므로 pos_weight=1.0
    - 리샘플링이 없으면(ClassWeight) pos_weight = (다수/소수)
    """
    method_l = str(method).strip().lower()
    ratio = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("BorderlineSMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("SMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' -> ClassWeight(pos_weight)로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    """Recall >= recall_min을 만족하는 가장 큰(엄격한) threshold 반환. 없으면 None."""
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if rec >= recall_min:
            valid.append(round(thr, 2))
    if not valid:
        return None
    return max(valid)


# ============================================================
# 4. 데이터 로드
# ============================================================
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)
y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]
all_data     = pd.concat([train_full, test], ignore_index=True)

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

print(f"\n피처 수    : {len(use_features)}개")
print(f"전체 데이터: {len(all_data)}행  |  기업 수: {all_data[COMPANY_COL].nunique()}개")
print("=" * 70)


# ============================================================
# 조회할 기업 × 회계년도 목록 ← ★ 여기만 수정하면 됨
# ============================================================
QUERY_LIST = [
    {"회사명": "현대코퍼레이션(주)", "회계년도": 2021},
    {"회사명": "농협경제지주주식회사", "회계년도": 2021},
    {"회사명": "현대코퍼레이션(주)", "회계년도": 2024},
    {"회사명": "농협경제지주주식회사", "회계년도": 2024},
    # {"회사명": "기업명", "회계년도": 연도},
]


# ============================================================
# 5. 헬퍼 함수
# ============================================================
def make_model(pos_weight):
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


def get_sample(company, year):
    mask   = (all_data[COMPANY_COL] == company) & (all_data[YEAR_COL] == year)
    result = all_data[mask]
    if len(result) == 0:
        print(f"  ⚠️  [{company} / {year}] 데이터 없음")
        return None
    if len(result) > 1:
        print(f"  ⚠️  [{company} / {year}] 중복 {len(result)}행 → 첫 번째 행 사용")
    return result.iloc[[0]]


def fix_minus(fig):
    """유니코드 마이너스(−) → 일반 하이픈(-) 치환"""
    for ax in fig.get_axes():
        for t in ax.get_xticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()])
        for t in ax.get_yticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_yticklabels([t.get_text() for t in ax.get_yticklabels()])
        for txt in ax.texts:
            txt.set_text(txt.get_text().replace("\u2212", "-"))


# ============================================================
# 6. 모델 학습 (Method/SMOTE_Ratio 기반 리샘플링 적용)
#    + Test 기준 THRESHOLD (Recall >= RECALL_MIN) 산출
# ============================================================
print(f"\n{'='*70}")
print(f"[로컬 SHAP] FeatureSet: {FEATURE_SET}  |  피처 수: {len(use_features)}개")
print(f"{'='*70}")

save_sub = os.path.join(SHAP_SAVE_DIR, FEATURE_SET)
os.makedirs(save_sub, exist_ok=True)

imputer     = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(train_full[use_features]),
    columns=use_features
)
X_test_imp  = pd.DataFrame(
    imputer.transform(test[use_features]),
    columns=use_features
)

X_train_res, y_train_res, pos_weight = apply_resampling(
    X_train_imp, y_train_full, METHOD, SMOTE_RATIO
)
print(f"  리샘플링 적용({METHOD}, SMOTE_Ratio={SMOTE_RATIO}): "
      f"{len(X_train_imp)}행 -> {len(X_train_res)}행, pos_weight={pos_weight:.4f}")

model = make_model(pos_weight)
model.fit(X_train_res, y_train_res)

# Test 데이터 기준 threshold 산출
y_prob_test = model.predict_proba(X_test_imp)[:, 1]
found_threshold = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
THRESHOLD = found_threshold if found_threshold is not None else 0.01
print(f"  Test 기준 threshold (Recall >= {RECALL_MIN}) = {THRESHOLD:.2f}")

# ── SHAP explainer 준비 ───────────────────────────────────
explainer = shap.TreeExplainer(model)

# ── 전체 데이터 imputation ────────────────────────────────
all_feat_imp = pd.DataFrame(
    imputer.transform(all_data[use_features]),
    columns=use_features,
    index=all_data.index
)


# ============================================================
# 7. 쿼리별 로컬 SHAP 분석
# ============================================================
for query in QUERY_LIST:

    company = query["회사명"]
    year    = query["회계년도"]

    print(f"\n  [{company} / {year}년]")

    sample_raw = get_sample(company, year)
    if sample_raw is None:
        continue

    sample_idx = sample_raw.index[0]
    X_sample   = all_feat_imp.loc[[sample_idx], use_features]

    y_true = sample_raw[TARGET_COL].values[0]
    y_prob = model.predict_proba(X_sample)[0, 1]
    y_pred = int(y_prob >= THRESHOLD)

    print(f"    실제 라벨 : {'부실(1)' if y_true == 1 else '정상(0)'}  |  "
          f"예측 확률 : {y_prob:.4f}  |  "
          f"예측 라벨 : {'부실(1)' if y_pred == 1 else '정상(0)'}")

    sv           = explainer(X_sample)
    shap_vals_1d = sv.values[0]
    base_value   = sv.base_values[0]

    safe_company = company.replace("/", "_").replace(" ", "_")
    save_company = os.path.join(save_sub, f"{safe_company}_{year}")
    os.makedirs(save_company, exist_ok=True)

    # ── [1] Waterfall Plot ────────────────────────────────
    mpl.rcParams["font.family"]        = "Malgun Gothic" \
                                         if platform.system() == "Windows" \
                                         else "AppleGothic"
    mpl.rcParams["axes.unicode_minus"] = False

    plt.figure(figsize=(10, max(6, len(use_features) * 0.28)))
    shap.plots.waterfall(sv[0], max_display=20, show=False)

    fix_minus(plt.gcf())

    plt.title(
        f"SHAP Waterfall — {company} ({year}년)\n"
        f"실제: {'부실' if y_true==1 else '정상'}  |  "
        f"예측확률: {y_prob:.4f}  |  임계값: {THRESHOLD:.2f}  |  "
        f"Method: {METHOD}",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_company, f"waterfall_{safe_company}_{year}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()
    print(f"    → waterfall_{safe_company}_{year}.png 저장 완료")

    # ── [2] SHAP 기여도 CSV ───────────────────────────────
    shap_row = pd.DataFrame({
        "Feature"    : use_features,
        "Value"      : X_sample.iloc[0].values,
        "SHAP_Value" : shap_vals_1d,
    })
    shap_row["Direction"] = shap_row["SHAP_Value"].apply(
        lambda x: "부실기여(+)" if x > 0 else "정상기여(-)"
    )
    shap_row["abs_SHAP"] = shap_row["SHAP_Value"].abs()
    shap_row = shap_row.sort_values(
        "abs_SHAP", ascending=False
    ).reset_index(drop=True)
    shap_row["Rank"]       = shap_row.index + 1
    shap_row.insert(0, "회계년도", year)
    shap_row.insert(0, "회사명",   company)
    shap_row["base_value"] = round(base_value, 6)
    shap_row["pred_prob"]  = round(y_prob, 6)
    shap_row["pred_label"] = y_pred
    shap_row["true_label"] = int(y_true)
    shap_row["threshold"]  = THRESHOLD
    shap_row["Method"]     = METHOD
    shap_row["SMOTE_Ratio"] = SMOTE_RATIO

    shap_row = shap_row[[
        "회사명", "회계년도",
        "Rank", "Feature", "Value", "SHAP_Value",
        "Direction", "abs_SHAP",
        "base_value", "pred_prob", "pred_label",
        "true_label", "threshold", "Method", "SMOTE_Ratio"
    ]]
    shap_row.to_csv(
        os.path.join(save_company, f"shap_local_{safe_company}_{year}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"    → shap_local_{safe_company}_{year}.csv 저장 완료")

    # 상위 5개 출력
    print(f"\n    [상위 5개 기여 피처]")
    print(shap_row[[
        "Rank", "Feature", "Value", "SHAP_Value", "Direction"
    ]].head(5).to_string(index=False))


print("\n" + "=" * 70)
print("로컬 SHAP 분석 완료 (Top10 1위 조합 기준)")
print(f"  FeatureSet: {FEATURE_SET} | Method: {METHOD} | SMOTE_Ratio: {SMOTE_RATIO} | "
      f"Threshold: {THRESHOLD:.2f}")
print(f"  저장 위치: {save_sub}")
print("=" * 70)

Top10 1위 조합
  FeatureSet  : top65_dedup52
  FeatureFile : lasso_features_top65--52.csv
  Method      : ClassWeight
  SMOTE_Ratio : -
  Model       : XGBoost

피처 수    : 52개
전체 데이터: 39908행  |  기업 수: 6106개

[로컬 SHAP] FeatureSet: top65_dedup52  |  피처 수: 52개
  리샘플링 적용(ClassWeight, SMOTE_Ratio=-): 28111행 -> 28111행, pos_weight=25.6455
  Test 기준 threshold (Recall >= 0.9) = 0.34

  [현대코퍼레이션(주) / 2021년]
    실제 라벨 : 정상(0)  |  예측 확률 : 0.0043  |  예측 라벨 : 정상(0)
    → waterfall_현대코퍼레이션(주)_2021.png 저장 완료
    → shap_local_현대코퍼레이션(주)_2021.csv 저장 완료

    [상위 5개 기여 피처]
 Rank                 Feature    Value  SHAP_Value Direction
    1           총자본영업이익률_diff 0.946766   -1.996689   정상기여(-)
    2             영업이익률_ratio 0.781534   -1.036776   정상기여(-)
    3                 금융비용부담률 0.010241   -0.818683   정상기여(-)
    4 금융비용대매출액_ratio_industry 0.011437   -0.563602   정상기여(-)
    5                   자본잠식률 0.956641   -0.208537   정상기여(-)

  [농협경제지주주식회사 / 2021년]
    실제 라벨 : 정상(0)  |  예측 확률 : 0.2353  |  예측 라벨 : 정상(0)